In [8]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tsai.all import *
from tsai.inference import load_learner
from sklearn.preprocessing import MinMaxScaler
import pdb
from IPython.display import clear_output as clear


In [9]:
#### 训练并保存模型

# ts = get_forecasting_time_series("Sunspots").values
df = pd.read_csv('Detroit_ Become Human.csv')
sc = MinMaxScaler(feature_range=(0, 1))
ts = sc.fit_transform(df['Owners'].values.reshape(-1, 1)).astype('float64')
window_length = 8 # 使用多少个时间步长来预测下一个
horizon = 1       # 预测未来的时间步长数
pred_len = 14 
X, y = SlidingWindow(window_length, horizon=horizon)(ts)
splits = TimeSplitter(pred_len)(y) 
tfms = [None, TSForecasting()]
batch_tfms = TSStandardize()
#"RNN", "LSTM", "GRU", "MLP", "FCN", 'ResNet', 'LSTM_FCN', 'GRU_FCN', 
# 'mWDN', 'TCN', 'MLSTM_FCN', 'InceptionTime',  'XceptionTime', 
# 'ResCNN',  'OmniScaleCNN', 'TST',  'TSiT',  'XCM', 'gMLP',  'TSSequencerPlus', 
# 这些不行(可能要修改默认参数 )：'Rocket', 'TabModel', 'TabTransformer', 'MiniRocket','TSPerceiver', 'GatedTabTransformer', 'PatchTST'
for models_name in ["RNN", "LSTM", "GRU", "MLP", "FCN", 'ResNet', 'LSTM_FCN', 'GRU_FCN', 
                    'mWDN', 'TCN', 'MLSTM_FCN', 'InceptionTime',  'XceptionTime',  'ResCNN',  
                    'OmniScaleCNN', 'TST',  'TSiT',  'XCM', 'gMLP',  'TSSequencerPlus', ]:
    fcst = TSForecaster(X, y, splits=splits, path='models_Detroit', tfms=tfms, batch_tfms=batch_tfms, bs=512, arch=models_name, metrics=[mae, mse, mape, rmse], cbs=ShowGraph())
    fcst.fit_one_cycle(100, 1e-3)
    fcst.export("Detroit_"+models_name+".pkl")

    clear()

In [ ]:
#### 测试并保存数据

for models_name in ["RNN", "LSTM", "GRU", "MLP", "FCN", 'ResNet', 'LSTM_FCN', 'GRU_FCN', 
                    'mWDN', 'TCN', 'MLSTM_FCN', 'InceptionTime',  'XceptionTime',  'ResCNN',  
                    'OmniScaleCNN', 'TST',  'TSiT',  'XCM', 'gMLP',  'TSSequencerPlus', ]:

    fcst = load_learner("models_Detroit/"+"Detroit_"+models_name+".pkl", cpu=False)
    raw_preds, target, preds = fcst.get_X_preds(X[splits[1]], y[splits[1]])
    target = sc.inverse_transform(np.array(target).reshape(-1, 1)).astype('int').flatten()
    preds = sc.inverse_transform(np.array(preds).reshape(-1, 1)).astype('int').flatten()

    pd.DataFrame({'date': df['date'].values[-pred_len:], 'Owners': preds}).to_csv('results_Detroit/Detroit_'+models_name+'preds_'+str(pred_len)+'.csv', index=False)
    # plt.plot(target)
    # plt.plot(preds)
    # plt.suptitle('tsai Prediction', )
    # plt.legend(['target', 'preds']) # , loc='upper left'
    # # plt.xticks(range(0, len(testY_index), 10), testY_index[::10], rotation=75)
    # plt.savefig('figs/Detroit_'+models_name+'_Predictions.png', bbox_inches='tight')
    # plt.show()

In [ ]:
#### metrics

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def get_metric(actual, predicted):
    # 计算MAE
    mae = mean_absolute_error(actual, predicted)
    # 计算MAPE
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    # 计算MSE
    mse = mean_squared_error(actual, predicted)
    # 计算RMSE
    rmse = np.sqrt(mse)
    # 计算R^2
    r2 = r2_score(actual, predicted)
    return mae, mape, mse, rmse, r2

df_t = pd.read_csv('results_Detroit/target.csv')['Owners']
df_9 = pd.read_csv('results_Detroit/trend_9.csv')['Owners']

model_list = ["RNN", "LSTM", "GRU", "MLP", "FCN", 'ResNet', 'LSTM_FCN', 'GRU_FCN', 
                    'mWDN', 'TCN', 'MLSTM_FCN', 'InceptionTime',  'XceptionTime',  'ResCNN',  
                    'OmniScaleCNN', 'TST',  'TSiT',  'XCM', 'gMLP',  'TSSequencerPlus']


for models_name in model_list:
    df = pd.read_csv('results_Detroit/Detroit_'+models_name+'preds_14.csv')['Owners']
    mae, mape, mse, rmse, r2 = get_metric(df, df_t)
    print(mae, mape, mse, rmse, r2)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import savemat

df_t = pd.read_csv('results_Detroit/target.csv')
date = df_t['date']
df_t = df_t['Owners']
df_9 = pd.read_csv('results_Detroit/trend_9.csv')['Owners']

model_list = ["RNN", "LSTM", "GRU", "MLP", "FCN", 'ResNet', 'LSTM_FCN', 'GRU_FCN', 
                    'mWDN', 'TCN', 'MLSTM_FCN', 'InceptionTime',  'XceptionTime',  'ResCNN',  
                    'OmniScaleCNN', 'TST',  'TSiT',  'XCM', 'gMLP',  'TSSequencerPlus']

plt.plot(date, df_t)
plt.plot(date, df_9)

for models_name in model_list:
    df = pd.read_csv('results_Detroit/Detroit_'+models_name+'preds_14.csv')['Owners']
    savemat('MAT_Detroit/'+models_name+".mat", {models_name: df})  # 方面后面用matlab画图，matlab处理 
    plt.plot(date, df)

plt.xticks(date, rotation=60)
plt.legend(['target', 'trend_9']+model_list)
plt.show()
